In [1]:
import numpy as np
import torch
import scanpy as sc
import anndata as ad
import os
import pandas as pd
from utils.preprocess import *

In [2]:
import os
import sys
    
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc
import torch
import torchsde
from torchdyn.core import NeuralODE
from tqdm import tqdm

from torchcfm.conditional_flow_matching import *
from torchcfm.models import MLP
from torchcfm.utils import plot_trajectories, torch_wrapper
from simulate.simulate import *
from omegaconf import OmegaConf
from utils.hydra import *
from datasets.process import *
from scripts.run_model import *
from eval.eval import *

In [3]:
import matplotlib.pyplot as plt
%matplotlib inline

In [4]:
%reload_ext autoreload
%autoreload 2

In [5]:
############################################################

In [6]:
### SETTINGS ###

from collections import namedtuple
Holdout = namedtuple('Holdout', ['t'])
holdout = Holdout(3)

config = load_config()

OmegaConf.set_struct(config, False)
config.pc_dim = 100
adata = process_data(pc_dim=config.pc_dim, data="cite")



config.num_classes = adata.obs['cell_type'].nunique()

config.metric = "cfm"
config.finsler.use = False
config.finsler.lamb = 3.0

config.K = 150
config.kappa = 1.5
config.classifier_max_epochs = 2
config.metric_max_epochs = 2
# config.embed_max_epochs = 2
config.flow_max_epochs = 2

project = "cite"

In [7]:
#TODO: fix + make ctrl make sense
test_bool = adata.obs['timepoint'] == holdout.t
adata_train = adata[~test_bool]
adata_test = adata[test_bool]

In [8]:
############################################################

In [9]:
tree = adata.uns['tree']
timepoints = sorted(adata.obs['timepoint'].unique().tolist())
singleton_dataloader = build_singleton_dataloader(config, adata_train)
paired_dataloader = build_paired_dataloader(config, adata_train)
classifier_model, metric_model, embed_model, flow_model = run_full_model(config=config,
                                                                         project=project,
                                                                         singleton_dataloader=singleton_dataloader,
                                                                         paired_dataloader=paired_dataloader,
                                                                         timepoints=timepoints,
                                                                         tree=tree)

DEBUG: turned off random v in embed loss
DEBUG: skip embed for OT
Running phase classifier:.......


wandb: Currently logged in as: az831 (az831-new-york-genome-center) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: Using wandb-core as the SDK backend.  Please refer to https://wandb.me/wandb-core for more information.


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
You are using a CUDA device ('NVIDIA GeForce RTX 4080 SUPER') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name           | Type           | Params | Mode 
----------------------------------------------------

epoch,▁▁▁▁▁▁▁▁▁▁▁███████████
train_ce,█▇▇▆▆▆▅▅▅▄▄▄▃▃▃▃▂▂▂▂▁▁
train_kl,▂▂▂▁▁▂▁▁▁▂▂▂▂▃▄▄▅▅▆▆██
train_loss,█▇▇▆▆▆▅▅▅▄▄▄▃▃▂▃▂▂▂▁▁▁
trainer/global_step,▁▁▂▂▂▃▃▃▄▄▄▅▅▅▆▆▆▇▇▇██
epoch,1
train_ce,1.30084
train_kl,0.25368
train_loss,1.31352
trainer/global_step,21


Running phase metric:.......


...

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name         | Type           | Params | Mode 
--------------------------------------------------------
0 | embed_net    | SimpleEmbedNet | 250 K  | train
1 | geo_net      | SinNet         | 292 K  | train
2 | metric_model | MetricNetCFM   | 0      | eval 
--------------------------------------------------------
543 K     Trainable params
0         Non-trainable params
543 K     Total params
2.176     Total estimated model params size (MB)
32        Modules in train 

epoch,▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▄▄▅▅▅▅▅▅▅▆▆▆▇▇▇▇▇▇████
train_loss_embed,█▇▇▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train_loss_geo,▄▆▄▄▅▂▇▅▆▆▄▃▅▃▃▅▄▁▃▆▂▁▂▆▃▇▆█▄▅▄▅▅▆▃▂▄▂▅▂
trainer/global_step,▁▁▂▂▂▂▂▂▂▃▃▃▄▄▄▄▄▅▅▅▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇███
epoch,2999
train_loss_embed,13.50926
train_loss_geo,8523.91504
trainer/global_step,2999


Running phase flow:.......


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/home/azweig/miniconda3/envs/spatialenv/lib/python3.10/site-packages/pytorch_lightning/loggers/wandb.py:397: There is a wandb run already in progress and newly created instances of `WandbLogger` will reuse this run. If this is not desired, call `wandb.finish()` before instantiating `WandbLogger`.
wandb: logging graph, to disable use `wandb.watch(log_graph=False)`
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]

  | Name        | Type              | Params | Mode 
----------------------------------------------------------
0 | flow_net    | SinNet            | 267 K  | train
1 | embed_model | EmbedNetTrainBase | 543 K  | eval 
----------------------------------------------------------
267 K     Trainable params
543 K     Non-trainable params
811 K     Total params
3.245     Total estimated model params size (MB)
16        Modules in train mode
34        Modules in eval mode
/home

epoch,▁█
train_loss,▁█
trainer/global_step,▁█
epoch,1
train_loss,172.20537
trainer/global_step,1


In [10]:
#fix the wandb.run.summary bug?
def remove_all_forward_hooks(model):
    for module in model.modules():
        module._forward_hooks.clear()

remove_all_forward_hooks(classifier_model)
remove_all_forward_hooks(metric_model)
remove_all_forward_hooks(embed_model)
remove_all_forward_hooks(flow_model)

In [14]:
t = holdout.t

print(predict(embed_model, adata, t, num_traj=6000, library="pot"))

tensor(44.2075)
